In [1]:
# ============================================================
# CBZ–SAC: Crystal structure generation and atom identification
# Code 1: Install packages and upload the CIF file
# ============================================================

!pip -q install pymatgen plotly pandas

from google.colab import files
import os
import numpy as np
import pandas as pd

from pymatgen.core import Structure, Element
from pymatgen.io.cif import CifParser

print("Packages installed successfully.")
print("Now upload the original CBZ–SAC CIF file.")

uploaded = files.upload()

cif_files = [
    filename for filename in uploaded
    if filename.lower().endswith((".cif", ".mcif"))
]

if len(cif_files) == 0:
    raise ValueError(
        "No CIF file detected. Please upload the CBZ–SAC CIF file."
    )

CIF_FILE = cif_files[0]

print("\nSelected CIF file:", CIF_FILE)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Packages installed successfully.
Now upload the original CBZ–SAC CIF file.


Saving CHGNet_relaxed (8).cif to CHGNet_relaxed (8).cif

Selected CIF file: CHGNet_relaxed (8).cif


In [2]:

from pymatgen.io.cif import CifParser
import numpy as np
import pandas as pd

# Load the uploaded original CBZ–SAC CIF
parser = CifParser(CIF_FILE)
structures = parser.parse_structures(primitive=False)

if len(structures) == 0:
    raise ValueError("No crystal structure could be read from the CIF file.")

structure = structures[0]

# Basic crystal information
print("CIF file:", CIF_FILE)
print("Number of atomic sites:", len(structure))
print("Formula:", structure.composition.reduced_formula)
print("Lattice parameters (Å and degrees):")
print(f"a = {structure.lattice.a:.4f} Å")
print(f"b = {structure.lattice.b:.4f} Å")
print(f"c = {structure.lattice.c:.4f} Å")
print(f"alpha = {structure.lattice.alpha:.3f}°")
print(f"beta  = {structure.lattice.beta:.3f}°")
print(f"gamma = {structure.lattice.gamma:.3f}°")
print(f"Unit-cell volume = {structure.lattice.volume:.3f} Å³")

# Preserve and inspect the CIF site's order
site_data = []
for i, site in enumerate(structure):
    site_data.append({
        "Atom_index": i,
        "Element": site.specie.symbol,
        "Fractional_x": site.frac_coords[0],
        "Fractional_y": site.frac_coords[1],
        "Fractional_z": site.frac_coords[2],
        "Cartesian_x_A": site.coords[0],
        "Cartesian_y_A": site.coords[1],
        "Cartesian_z_A": site.coords[2]
    })

atom_table = pd.DataFrame(site_data)

print("\nFirst 15 atomic sites:")
display(atom_table.head(15))

print("\nElement counts:")
display(atom_table["Element"].value_counts().rename_axis("Element").to_frame("Count"))


CIF file: CHGNet_relaxed (8).cif
Number of atomic sites: 384
Formula: H19C22N3O4
Lattice parameters (Å and degrees):
a = 27.4401 Å
b = 7.6430 Å
c = 21.0459 Å
alpha = 90.000°
beta  = 119.256°
gamma = 90.000°
Unit-cell volume = 3850.798 Å³

First 15 atomic sites:


,Atom_index,Element,Fractional_x,Fractional_y,Fractional_z,Cartesian_x_A,Cartesian_y_A,Cartesian_z_A
0,0,H,0.753762,0.764026,0.692402,18.044943,5.839436,4.463880
1,1,H,0.246238,0.764027,0.807598,5.894914,5.839438,13.694406
2,2,H,0.253761,0.264026,0.692402,6.075014,2.017946,11.169132
3,3,H,0.746238,0.264026,0.807598,17.864843,2.017947,6.989151
4,4,H,0.246239,0.235974,0.307598,5.894915,1.803542,3.171475
5,5,H,0.753761,0.235974,0.192402,18.044940,1.803542,-6.059051
6,6,H,0.746238,0.735973,0.307598,17.864843,5.625027,-3.533780
7,7,H,0.253761,0.735974,0.192402,6.075014,5.625032,0.646206
8,8,H,0.481381,0.375459,0.551092,11.524188,2.869623,5.142646
9,9,H,0.518619,0.375459,0.948908,12.415671,2.869624,13.015635



Element counts:


,Count
Element,
C,176
H,152
O,32
N,24


In [3]:

import plotly.graph_objects as go
from pymatgen.core import Element

# Element colors for visualization
element_colors = {
    "C": "#606060",
    "H": "#D9D9D9",
    "N": "#305CDE",
    "O": "#E53935",
    "S": "#E0B323"
}

fig = go.Figure()

# Add each element as a separate trace
for element in sorted(set(site.specie.symbol for site in structure)):
    indices = [
        i for i, site in enumerate(structure)
        if site.specie.symbol == element
    ]

    coords = np.array([structure[i].coords for i in indices])

    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode="markers+text",
        name=element,
        text=[str(i) for i in indices],
        textposition="top center",
        marker=dict(
            size=5 if element != "H" else 3,
            color=element_colors.get(element, "#888888"),
            opacity=0.9
        ),
        hovertext=[
            f"Index: {i}<br>Element: {element}"
            for i in indices
        ],
        hoverinfo="text"
    ))

fig.update_layout(
    title="CBZ–SAC CIF: Atomic-site indices (0-based)",
    scene=dict(
        xaxis_title="X (Å)",
        yaxis_title="Y (Å)",
        zaxis_title="Z (Å)",
        aspectmode="data"
    ),
    width=1000,
    height=750,
    legend_title="Element",
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()


In [4]:

# Install Open Babel Python bindings
!pip -q install openbabel-wheel

import numpy as np
import plotly.graph_objects as go
from pymatgen.io.cif import CifParser
from openbabel import openbabel as ob

# Convert the CIF to an Open Babel molecule
conv = ob.OBConversion()
conv.SetInAndOutFormats("cif", "mol")

mol = ob.OBMol()
if not conv.ReadFile(mol, CIF_FILE):
    raise RuntimeError(
        "Open Babel could not read the CIF. "
        "Check the file and format support."
    )

# Add hydrogens only if needed by the imported molecule
mol.ConnectTheDots()
mol.PerceiveBondOrders()

# Extract coordinates and bonds
atoms = []
for atom in ob.OBMolAtomIter(mol):
    atoms.append({
        "index": atom.GetIdx() - 1,
        "element": ob.GetSymbol(atom.GetAtomicNum()),
        "x": atom.GetX(),
        "y": atom.GetY(),
        "z": atom.GetZ()
    })

bonds = []
for bond in ob.OBMolBondIter(mol):
    bonds.append((
        bond.GetBeginAtomIdx() - 1,
        bond.GetEndAtomIdx() - 1,
        bond.GetBondOrder()
    ))

# Element colors and ball sizes
colors = {
    "C": "#404040",
    "H": "#F2F2F2",
    "N": "#305CDE",
    "O": "#E53935",
    "S": "#E0B323"
}
sizes = {"C": 9, "H": 5, "N": 9, "O": 9, "S": 11}

fig = go.Figure()

# Draw bonds as line segments
for i, j, order in bonds:
    a, b = atoms[i], atoms[j]
    fig.add_trace(go.Scatter3d(
        x=[a["x"], b["x"]],
        y=[a["y"], b["y"]],
        z=[a["z"], b["z"]],
        mode="lines",
        line=dict(color="#A0A0A0", width=5),
        hoverinfo="skip",
        showlegend=False
    ))

# Draw atoms as balls
for element in colors:
    selected = [a for a in atoms if a["element"] == element]
    if not selected:
        continue

    fig.add_trace(go.Scatter3d(
        x=[a["x"] for a in selected],
        y=[a["y"] for a in selected],
        z=[a["z"] for a in selected],
        mode="markers",
        name=element,
        marker=dict(
            size=sizes[element],
            color=colors[element],
            line=dict(color="#333333", width=0.5),
            opacity=1
        ),
        text=[
            f'{a["element"]}, atom index {a["index"]}'
            for a in selected
        ],
        hoverinfo="text"
    ))

fig.update_layout(
    title="CBZ–SAC: Chem3D-style Ball-and-Stick View",
    scene=dict(
        xaxis_title="X (Å)",
        yaxis_title="Y (Å)",
        zaxis_title="Z (Å)",
        aspectmode="data"
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()
